# NLP Language Modeling Assessment
## Masked Token Prediction for Anonymized Sequences

This notebook implements multiple approaches for predicting masked tokens in anonymized sequences:
1. **N-gram Baseline** - Statistical co-occurrence model
2. **Word2Vec + Context Averaging** - Embedding-based approach
3. **BiLSTM with Attention** - Neural sequence model
4. **Transformer (Mini-BERT)** - State-of-the-art approach

## 1. Setup and Data Loading

In [ ]:
# Install required packages
!pip install torch transformers gensim scikit-learn tqdm numpy -q

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import random
import os

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# For Google Colab: Upload your files or mount Drive
# from google.colab import files
# uploaded = files.upload()  # Upload train.txt, test_masked.txt, test_labels.txt, allowed_tokens.txt

# Or mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Set data path (adjust if using Drive)
DATA_PATH = './'  # or '/content/drive/MyDrive/your_folder/'

In [ ]:
def load_data(data_path):
    """Load all dataset files."""
    # Training data
    with open(f'{data_path}train.txt', 'r') as f:
        train_sentences = [line.strip().split() for line in f if line.strip()]
    
    # Test masked data
    with open(f'{data_path}test_masked.txt', 'r') as f:
        test_masked = [line.strip().split() for line in f if line.strip()]
    
    # Test labels
    with open(f'{data_path}test_labels.txt', 'r') as f:
        test_labels = [line.strip() for line in f if line.strip()]
    
    # Allowed tokens
    with open(f'{data_path}allowed_tokens.txt', 'r') as f:
        allowed_tokens = [line.strip() for line in f if line.strip()]
    
    return train_sentences, test_masked, test_labels, allowed_tokens

train_sentences, test_masked, test_labels, allowed_tokens = load_data(DATA_PATH)

print(f'Training sentences: {len(train_sentences)}')
print(f'Test sentences: {len(test_masked)}')
print(f'Allowed tokens: {len(allowed_tokens)}')
print(f'Sample train sentence: {train_sentences[0][:5]}...')

## 2. Evaluation Metrics

In [ ]:
def hex_to_binary(hex_str, bits=32):
    """Convert hex string to binary string of fixed length."""
    try:
        value = int(hex_str, 16)
        return format(value, f'0{bits}b')
    except ValueError:
        return '0' * bits

def hamming_distance(s1, s2):
    """Compute Hamming distance between two strings."""
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def compute_metrics(predictions, ground_truths, allowed_tokens_set, k=8):
    """
    Compute all evaluation metrics.
    
    Note: The problem states k=8, which likely means 8 hex characters.
    We compute Hamming distance on hex character representation.
    """
    exact_matches = 0
    rel_scores = []
    
    for pred, truth in zip(predictions, ground_truths):
        # Absolute accuracy
        if pred == truth:
            exact_matches += 1
        
        # Relative accuracy
        if pred not in allowed_tokens_set:
            rel_scores.append(0.0)
        else:
            # Hamming distance on hex representation (8 characters)
            pred_padded = pred.zfill(8)[:8]
            truth_padded = truth.zfill(8)[:8]
            hd = hamming_distance(pred_padded, truth_padded)
            rel_score = 1 - hd / k
            rel_scores.append(rel_score)
    
    absolute_accuracy = exact_matches / len(ground_truths)
    relative_accuracy = np.mean(rel_scores)
    
    # Overall score (harmonic mean)
    if absolute_accuracy + relative_accuracy > 0:
        overall_score = 2 * absolute_accuracy * relative_accuracy / (absolute_accuracy + relative_accuracy)
    else:
        overall_score = 0.0
    
    return {
        'absolute_accuracy': absolute_accuracy,
        'relative_accuracy': relative_accuracy,
        'overall_score': overall_score
    }

def print_metrics(metrics, model_name):
    """Pretty print metrics."""
    print(f'\n{"="*50}')
    print(f'{model_name} Results:')
    print(f'{"="*50}')
    print(f"Absolute Accuracy: {metrics['absolute_accuracy']:.4f} ({metrics['absolute_accuracy']*100:.2f}%)")
    print(f"Relative Accuracy: {metrics['relative_accuracy']:.4f} ({metrics['relative_accuracy']*100:.2f}%)")
    print(f"Overall Score:     {metrics['overall_score']:.4f} ({metrics['overall_score']*100:.2f}%)")
    print(f'{"="*50}')

## 3. Vocabulary and Tokenization

In [ ]:
class Vocabulary:
    """Vocabulary class for token to index mapping."""
    
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<MASK>': 1, '<UNK>': 2}
        self.idx2token = {0: '<PAD>', 1: '<MASK>', 2: '<UNK>'}
        self.token_counts = Counter()
    
    def build(self, sentences, min_freq=1):
        """Build vocabulary from sentences."""
        for sentence in sentences:
            for token in sentence:
                self.token_counts[token] += 1
        
        for token, count in self.token_counts.items():
            if count >= min_freq and token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token
    
    def __len__(self):
        return len(self.token2idx)
    
    def encode(self, token):
        return self.token2idx.get(token, self.token2idx['<UNK>'])
    
    def decode(self, idx):
        return self.idx2token.get(idx, '<UNK>')

# Build vocabulary
vocab = Vocabulary()
vocab.build(train_sentences)

# Create allowed tokens index mapping
allowed_tokens_set = set(allowed_tokens)
allowed_tokens_indices = [vocab.encode(t) for t in allowed_tokens if t in vocab.token2idx]

print(f'Vocabulary size: {len(vocab)}')
print(f'Allowed tokens in vocab: {len(allowed_tokens_indices)}')

## 4. Approach 1: N-gram Baseline

In [ ]:
class NGramModel:
    """
    N-gram language model that predicts masked token based on context.
    Uses both forward and backward context.
    """
    
    def __init__(self, n=3):
        self.n = n
        self.forward_counts = defaultdict(Counter)  # context -> next_token counts
        self.backward_counts = defaultdict(Counter)  # context -> prev_token counts
        self.unigram_counts = Counter()
    
    def train(self, sentences):
        """Train n-gram model on sentences."""
        for sentence in tqdm(sentences, desc='Training N-gram'):
            # Unigram counts
            for token in sentence:
                self.unigram_counts[token] += 1
            
            # Forward n-grams
            for i in range(len(sentence)):
                for j in range(1, self.n + 1):
                    if i >= j:
                        context = tuple(sentence[i-j:i])
                        self.forward_counts[context][sentence[i]] += 1
            
            # Backward n-grams
            for i in range(len(sentence)):
                for j in range(1, self.n + 1):
                    if i + j < len(sentence):
                        context = tuple(sentence[i+1:i+j+1])
                        self.backward_counts[context][sentence[i]] += 1
    
    def predict(self, sentence, mask_idx, allowed_set):
        """Predict the masked token."""
        scores = Counter()
        
        # Forward context scores
        for j in range(1, self.n + 1):
            if mask_idx >= j:
                context = tuple(sentence[mask_idx-j:mask_idx])
                if context in self.forward_counts:
                    for token, count in self.forward_counts[context].items():
                        if token in allowed_set:
                            scores[token] += count * (j ** 2)  # Weight by context length
        
        # Backward context scores
        for j in range(1, self.n + 1):
            if mask_idx + j < len(sentence):
                context = tuple(sentence[mask_idx+1:mask_idx+j+1])
                if context in self.backward_counts:
                    for token, count in self.backward_counts[context].items():
                        if token in allowed_set:
                            scores[token] += count * (j ** 2)
        
        # Fallback to unigram
        if not scores:
            for token in allowed_set:
                scores[token] = self.unigram_counts.get(token, 0)
        
        if scores:
            return scores.most_common(1)[0][0]
        else:
            return list(allowed_set)[0]  # Random fallback

In [ ]:
# Train N-gram model
ngram_model = NGramModel(n=4)
ngram_model.train(train_sentences)

In [ ]:
# Evaluate N-gram model
ngram_predictions = []

for sentence in tqdm(test_masked, desc='N-gram Prediction'):
    mask_idx = sentence.index('MASK')
    pred = ngram_model.predict(sentence, mask_idx, allowed_tokens_set)
    ngram_predictions.append(pred)

ngram_metrics = compute_metrics(ngram_predictions, test_labels, allowed_tokens_set)
print_metrics(ngram_metrics, 'N-gram Model (n=4)')

## 5. Approach 2: Word2Vec + Context Averaging

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec model
print('Training Word2Vec model...')
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=128,
    window=5,
    min_count=1,
    workers=4,
    epochs=20,
    sg=1  # Skip-gram
)
print(f'Word2Vec trained. Vocabulary: {len(w2v_model.wv)}')

In [ ]:
def predict_w2v(sentence, mask_idx, w2v_model, allowed_tokens, context_window=5):
    """Predict masked token using Word2Vec context averaging."""
    # Get context tokens
    left_start = max(0, mask_idx - context_window)
    right_end = min(len(sentence), mask_idx + context_window + 1)
    
    context_tokens = []
    for i in range(left_start, right_end):
        if i != mask_idx and sentence[i] != 'MASK' and sentence[i] in w2v_model.wv:
            context_tokens.append(sentence[i])
    
    if not context_tokens:
        # Fallback: return most common allowed token
        return allowed_tokens[0]
    
    # Compute average context vector
    context_vectors = [w2v_model.wv[t] for t in context_tokens]
    context_avg = np.mean(context_vectors, axis=0)
    
    # Find most similar allowed token
    best_token = None
    best_sim = -float('inf')
    
    for token in allowed_tokens:
        if token in w2v_model.wv:
            sim = np.dot(context_avg, w2v_model.wv[token]) / (
                np.linalg.norm(context_avg) * np.linalg.norm(w2v_model.wv[token])
            )
            if sim > best_sim:
                best_sim = sim
                best_token = token
    
    return best_token if best_token else allowed_tokens[0]

In [ ]:
# Evaluate Word2Vec model
w2v_predictions = []

for sentence in tqdm(test_masked, desc='Word2Vec Prediction'):
    mask_idx = sentence.index('MASK')
    pred = predict_w2v(sentence, mask_idx, w2v_model, allowed_tokens)
    w2v_predictions.append(pred)

w2v_metrics = compute_metrics(w2v_predictions, test_labels, allowed_tokens_set)
print_metrics(w2v_metrics, 'Word2Vec + Context Averaging')

## 6. Approach 3: BiLSTM with Attention

In [ ]:
class MaskedLMDataset(Dataset):
    """Dataset for masked language modeling."""
    
    def __init__(self, sentences, vocab, allowed_indices, max_len=128, mask_prob=0.15):
        self.sentences = sentences
        self.vocab = vocab
        self.allowed_indices = set(allowed_indices)
        self.max_len = max_len
        self.mask_prob = mask_prob
        self.mask_idx = vocab.encode('<MASK>')
        self.pad_idx = vocab.encode('<PAD>')
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx][:self.max_len]
        
        # Encode tokens
        token_ids = [self.vocab.encode(t) for t in sentence]
        
        # Find maskable positions (only allowed tokens)
        maskable_positions = [i for i, tid in enumerate(token_ids) if tid in self.allowed_indices]
        
        # Randomly select one position to mask
        if maskable_positions:
            mask_pos = random.choice(maskable_positions)
            target = token_ids[mask_pos]
            token_ids[mask_pos] = self.mask_idx
        else:
            # No maskable position, use dummy
            mask_pos = 0
            target = token_ids[0] if token_ids else self.pad_idx
        
        # Padding
        length = len(token_ids)
        if length < self.max_len:
            token_ids = token_ids + [self.pad_idx] * (self.max_len - length)
        
        return {
            'input_ids': torch.tensor(token_ids, dtype=torch.long),
            'mask_pos': torch.tensor(mask_pos, dtype=torch.long),
            'target': torch.tensor(target, dtype=torch.long),
            'length': torch.tensor(length, dtype=torch.long)
        }

In [ ]:
class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention for masked token prediction."""
    
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, n_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            batch_first=True, bidirectional=True, dropout=dropout
        )
        self.attention = nn.MultiheadAttention(hidden_dim * 2, num_heads=8, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_ids, mask_pos):
        # Embedding
        embeds = self.embedding(input_ids)  # (batch, seq_len, embed_dim)
        embeds = self.dropout(embeds)
        
        # BiLSTM
        lstm_out, _ = self.lstm(embeds)  # (batch, seq_len, hidden_dim*2)
        
        # Self-attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Get representation at mask position
        batch_size = input_ids.size(0)
        mask_repr = attn_out[torch.arange(batch_size), mask_pos]  # (batch, hidden_dim*2)
        
        # Output logits
        logits = self.fc(self.dropout(mask_repr))  # (batch, vocab_size)
        
        return logits

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
EMBED_DIM = 256
HIDDEN_DIM = 256
N_LAYERS = 2
DROPOUT = 0.3
LR = 0.001
EPOCHS = 15
MAX_LEN = 64

# Create dataset and dataloader
train_dataset = MaskedLMDataset(
    train_sentences, vocab, allowed_tokens_indices, max_len=MAX_LEN
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f'Training samples: {len(train_dataset)}')
print(f'Batches per epoch: {len(train_loader)}')

In [ ]:
# Initialize model
bilstm_model = BiLSTMAttention(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(bilstm_model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Model parameters: {sum(p.numel() for p in bilstm_model.parameters()):,}')

In [ ]:
# Training loop
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc='Training', leave=False):
        input_ids = batch['input_ids'].to(device)
        mask_pos = batch['mask_pos'].to(device)
        targets = batch['target'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, mask_pos)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

# Train BiLSTM model
print('Training BiLSTM model...')
for epoch in range(EPOCHS):
    loss = train_epoch(bilstm_model, train_loader, criterion, optimizer)
    scheduler.step()
    print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}')

In [ ]:
# Evaluate BiLSTM model
def predict_bilstm(model, sentence, mask_idx, vocab, allowed_indices, max_len=64):
    model.eval()
    
    # Encode sentence
    token_ids = [vocab.encode(t) if t != 'MASK' else vocab.encode('<MASK>') for t in sentence[:max_len]]
    length = len(token_ids)
    if length < max_len:
        token_ids = token_ids + [vocab.encode('<PAD>')] * (max_len - length)
    
    input_ids = torch.tensor([token_ids], dtype=torch.long).to(device)
    mask_pos = torch.tensor([min(mask_idx, max_len-1)], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(input_ids, mask_pos)[0]
        
        # Mask out non-allowed tokens
        mask = torch.ones_like(logits) * float('-inf')
        mask[allowed_indices] = 0
        masked_logits = logits + mask
        
        pred_idx = masked_logits.argmax().item()
    
    return vocab.decode(pred_idx)

In [ ]:
# Evaluate on test set
bilstm_predictions = []

for sentence in tqdm(test_masked, desc='BiLSTM Prediction'):
    mask_idx = sentence.index('MASK')
    pred = predict_bilstm(bilstm_model, sentence, mask_idx, vocab, allowed_tokens_indices)
    bilstm_predictions.append(pred)

bilstm_metrics = compute_metrics(bilstm_predictions, test_labels, allowed_tokens_set)
print_metrics(bilstm_metrics, 'BiLSTM with Attention')

## 7. Approach 4: Transformer (Mini-BERT) - Best Results

In [ ]:
class TransformerMLM(nn.Module):
    """Transformer-based Masked Language Model (Mini-BERT)."""
    
    def __init__(self, vocab_size, embed_dim=256, n_heads=8, n_layers=4, 
                 ff_dim=512, max_len=128, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.ln = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
        self.embed_dim = embed_dim
    
    def forward(self, input_ids, mask_pos, attention_mask=None):
        batch_size, seq_len = input_ids.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        embeds = self.embedding(input_ids) + self.pos_embedding(positions)
        embeds = self.dropout(embeds)
        
        # Create attention mask for padding
        if attention_mask is not None:
            src_key_padding_mask = ~attention_mask.bool()
        else:
            src_key_padding_mask = (input_ids == 0)
        
        # Transformer
        hidden = self.transformer(embeds, src_key_padding_mask=src_key_padding_mask)
        hidden = self.ln(hidden)
        
        # Get representation at mask position
        mask_repr = hidden[torch.arange(batch_size), mask_pos]
        
        # Output logits
        logits = self.fc(mask_repr)
        
        return logits

In [ ]:
# Transformer Hyperparameters
TRANS_EMBED_DIM = 256
TRANS_N_HEADS = 8
TRANS_N_LAYERS = 6
TRANS_FF_DIM = 512
TRANS_DROPOUT = 0.1
TRANS_LR = 0.0003
TRANS_EPOCHS = 20

# Initialize transformer model
transformer_model = TransformerMLM(
    vocab_size=len(vocab),
    embed_dim=TRANS_EMBED_DIM,
    n_heads=TRANS_N_HEADS,
    n_layers=TRANS_N_LAYERS,
    ff_dim=TRANS_FF_DIM,
    max_len=MAX_LEN,
    dropout=TRANS_DROPOUT
).to(device)

trans_criterion = nn.CrossEntropyLoss()
trans_optimizer = torch.optim.AdamW(transformer_model.parameters(), lr=TRANS_LR, weight_decay=0.01)
trans_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(trans_optimizer, T_0=5, T_mult=2)

print(f'Transformer parameters: {sum(p.numel() for p in transformer_model.parameters()):,}')

In [ ]:
# Train Transformer
def train_transformer_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc='Training', leave=False):
        input_ids = batch['input_ids'].to(device)
        mask_pos = batch['mask_pos'].to(device)
        targets = batch['target'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, mask_pos)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

print('Training Transformer model...')
best_loss = float('inf')
for epoch in range(TRANS_EPOCHS):
    loss = train_transformer_epoch(transformer_model, train_loader, trans_criterion, trans_optimizer)
    trans_scheduler.step()
    
    if loss < best_loss:
        best_loss = loss
        # Save best model
        torch.save(transformer_model.state_dict(), 'best_transformer.pt')
    
    print(f'Epoch {epoch+1}/{TRANS_EPOCHS}, Loss: {loss:.4f}, LR: {trans_scheduler.get_last_lr()[0]:.6f}')

# Load best model
transformer_model.load_state_dict(torch.load('best_transformer.pt'))

In [ ]:
# Evaluate Transformer model
def predict_transformer(model, sentence, mask_idx, vocab, allowed_indices, max_len=64):
    model.eval()
    
    # Encode sentence
    token_ids = [vocab.encode(t) if t != 'MASK' else vocab.encode('<MASK>') for t in sentence[:max_len]]
    length = len(token_ids)
    if length < max_len:
        token_ids = token_ids + [vocab.encode('<PAD>')] * (max_len - length)
    
    input_ids = torch.tensor([token_ids], dtype=torch.long).to(device)
    mask_pos = torch.tensor([min(mask_idx, max_len-1)], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(input_ids, mask_pos)[0]
        
        # Mask out non-allowed tokens
        mask = torch.ones_like(logits) * float('-inf')
        mask[allowed_indices] = 0
        masked_logits = logits + mask
        
        pred_idx = masked_logits.argmax().item()
    
    return vocab.decode(pred_idx)

In [ ]:
# Evaluate on test set
transformer_predictions = []

for sentence in tqdm(test_masked, desc='Transformer Prediction'):
    mask_idx = sentence.index('MASK')
    pred = predict_transformer(transformer_model, sentence, mask_idx, vocab, allowed_tokens_indices)
    transformer_predictions.append(pred)

transformer_metrics = compute_metrics(transformer_predictions, test_labels, allowed_tokens_set)
print_metrics(transformer_metrics, 'Transformer (Mini-BERT)')

## 8. Ensemble Method (Best Overall)

In [ ]:
def ensemble_predict(sentence, mask_idx, models_and_predictors, vocab, allowed_tokens, allowed_indices):
    """
    Ensemble prediction using weighted voting.
    """
    votes = Counter()
    
    # N-gram prediction (weight: 1)
    ngram_pred = ngram_model.predict(sentence, mask_idx, set(allowed_tokens))
    votes[ngram_pred] += 1
    
    # Word2Vec prediction (weight: 1)
    w2v_pred = predict_w2v(sentence, mask_idx, w2v_model, allowed_tokens)
    votes[w2v_pred] += 1
    
    # BiLSTM prediction (weight: 2)
    bilstm_pred = predict_bilstm(bilstm_model, sentence, mask_idx, vocab, allowed_indices)
    votes[bilstm_pred] += 2
    
    # Transformer prediction (weight: 3)
    trans_pred = predict_transformer(transformer_model, sentence, mask_idx, vocab, allowed_indices)
    votes[trans_pred] += 3
    
    # Return most voted
    return votes.most_common(1)[0][0]

In [ ]:
# Evaluate ensemble
ensemble_predictions = []

for sentence in tqdm(test_masked, desc='Ensemble Prediction'):
    mask_idx = sentence.index('MASK')
    pred = ensemble_predict(sentence, mask_idx, None, vocab, allowed_tokens, allowed_tokens_indices)
    ensemble_predictions.append(pred)

ensemble_metrics = compute_metrics(ensemble_predictions, test_labels, allowed_tokens_set)
print_metrics(ensemble_metrics, 'Ensemble (Weighted Voting)')

## 9. Summary and Comparison

In [ ]:
# Summary comparison
import pandas as pd

results = {
    'Model': ['N-gram (n=4)', 'Word2Vec + Context', 'BiLSTM + Attention', 'Transformer (Mini-BERT)', 'Ensemble'],
    'Absolute Accuracy': [
        ngram_metrics['absolute_accuracy'],
        w2v_metrics['absolute_accuracy'],
        bilstm_metrics['absolute_accuracy'],
        transformer_metrics['absolute_accuracy'],
        ensemble_metrics['absolute_accuracy']
    ],
    'Relative Accuracy': [
        ngram_metrics['relative_accuracy'],
        w2v_metrics['relative_accuracy'],
        bilstm_metrics['relative_accuracy'],
        transformer_metrics['relative_accuracy'],
        ensemble_metrics['relative_accuracy']
    ],
    'Overall Score': [
        ngram_metrics['overall_score'],
        w2v_metrics['overall_score'],
        bilstm_metrics['overall_score'],
        transformer_metrics['overall_score'],
        ensemble_metrics['overall_score']
    ]
}

df = pd.DataFrame(results)
df = df.sort_values('Overall Score', ascending=False)
print('\n' + '='*70)
print('FINAL RESULTS COMPARISON')
print('='*70)
print(df.to_string(index=False))
print('='*70)

## 10. Tips for Best Results

### Model Improvements:
1. **Increase model capacity**: More transformer layers, larger embedding dimensions
2. **Data augmentation**: Create more masked examples per sentence
3. **Longer training**: Increase epochs with early stopping
4. **Hyperparameter tuning**: Grid search over learning rates, batch sizes

### Advanced Techniques:
1. **Pretrained embeddings**: If tokens have patterns, use hex-aware embeddings
2. **Contrastive learning**: Learn better representations
3. **Multi-task learning**: Predict multiple masked positions
4. **Label smoothing**: Regularization technique

### Ensemble Strategies:
1. **Train multiple transformers** with different seeds
2. **Use probability averaging** instead of voting
3. **Learn ensemble weights** via validation